# Cal3f

A `Cal3f` represents a simplified 1-parameter camera calibration model {cite:p}`https://doi.org/10.1007/978-3-030-34372-9`[^f1]. This model includes one parameter for a shared focal length ($f_x = f_y = f$), zero skew ($s=0$), and a fixed principal point ($u_0, v_0$). It does not model lens distortion. The calibration matrix $K$ is defined as:

$$
K = \begin{bmatrix} f & 0 & u_0 \\ 0 & f & v_0 \\ 0 & 0 & 1 \end{bmatrix}
$$

The main purpose of this model is for conversions between image pixel coordinates $(u, v)$ in the image sensor frame and the normalized image coordinates $(x, y)$. The normalized image coordinates (also called intrinsic coordinates) are coordinates on a canonical image plane that is one unit of focal length away from the aperture point. In other words, given any 3D point $(x_c, y_c, z_c)$ in the camera frame, a `Cal3f` model can handle conversions between the normalized coordinates $(x, y) = \left(\frac{x_c}{z_c}, \frac{y_c}{z_c}\right)$ and the pixel coordinates $(u, v)$, where

$$
\begin{align*}
u &= f \cdot x + u_0 \\
v &= f \cdot y + v_0
\end{align*}
$$

<!-- The following will not be rendered as literal text by MyST -->
[^f1]: See ch. 2 $\S$ 2.1.4, pp. 57–58.

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/geometry/doc/Cal3f.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install --quiet gtsam-develop

In [8]:
import numpy as np
from gtsam import Cal3f, Point2

## Initialization

A `Cal3f` can be initialized in a couple of ways. Initializing with no arguments yields a calibration model with focal length $f=1$ and principal point at $(u_0,v_0)=(0,0)$; in other words, the calibration matrix becomes the identity matrix. You can also construct a model by specifying the focal length and principal point.

In [9]:
# Default constructor: f=1, u0=0, v0=0
cal_default = Cal3f()
print("Default constructor (f=1, u0=0, v0=0):")
print(cal_default)
print(f"f: {cal_default.f()}, u0: {cal_default.px()}, v0: {cal_default.py()}\n")

# From parameters: f, u0, v0
f, u0, v0 = 1500.0, 320.0, 240.0
cal_params = Cal3f(f, u0, v0)
print("From parameters (f, u0, v0):")
print(cal_params)
print(f"f: {cal_params.f()}, u0: {cal_params.px()}, v0: {cal_params.py()}\n")

Default constructor (f=1, u0=0, v0=0):
f: 1, px: 0, py: 0

f: 1.0, u0: 0.0, v0: 0.0

From parameters (f, u0, v0):
f: 1500, px: 320, py: 240

f: 1500.0, u0: 320.0, v0: 240.0



## Properties

### Main Parameters

The main parameters of the `Cal3f` model can be accessed using the following member functions:
- `f()`: Returns the effective focal length $f$.
- `px()`: Returns $u_0$, the x-coordinate of the principal point with respect to the image sensor frame.
- `py()`: Returns $v_0$, the y-coordinate of the principal point with respect to the image sensor frame.
- `vector()`: Returns a `numpy.ndarray` with 5 elements depicting a vectorized form of the calibration parameters. The returned 5-vector follow the form $(f, f, 0, u_0, v_0)$. 

In [10]:
f, u0, v0 = 1500.0, 320.0, 240.0
cal = Cal3f(f, u0, v0)

print("Calibration object properties:")
print(f"f: {cal.f()}")
print(f"u0: {cal.px()}")
print(f"v0: {cal.py()}")
print(f"Calibration vector [f]: {cal.vector()}")

Calibration object properties:
f: 1500.0
u0: 320.0
v0: 240.0
Calibration vector [f]: [1500. 1500.    0.  320.  240.]


### Derived Properties

`Cal3f` also has some member functions to retrieve useful derived properties:
- `K()`: Returns a `numpy.ndarray` with a dimension of $3\times 3$ representing the calibration matrix $K$ for the model.
- `inverse()`: Returns the inverted calibration matrix $K^{-1}$.

In [11]:
print(f"K matrix:\n{cal.K()}")
print(f"inv(K) matrix:\n{cal.inverse()}")

K matrix:
[[1.5e+03 0.0e+00 3.2e+02]
 [0.0e+00 1.5e+03 2.4e+02]
 [0.0e+00 0.0e+00 1.0e+00]]
inv(K) matrix:
[[ 6.66666667e-04 -0.00000000e+00 -2.13333333e-01]
 [ 0.00000000e+00  6.66666667e-04 -1.60000000e-01]
 [ 0.00000000e+00  0.00000000e+00  1.00000000e+00]]


## Basic Operations

`Cal3f` provides member functions to convert points between normalized image coordinates and pixel coordinates.

### `calibrate()`

The `calibrate(p)` member function converts a 2D point `p` in image pixel coordinates $(u, v)$ to normalized image coordinates $(x, y)$. This member function effectively implements the formula

$$
p_{\text{norm}} = K^{-1} \cdot p_{\text{pixels}}
$$

where $p_{\text{norm}}$ and $p_{\text{pixels}}$ are homogeneous representations of $(x, y)$ and $(u, v)$, respectively. 

In [14]:
f, u0, v0 = 1000.0, 320.0, 240.0
cal = Cal3f(f, u0, v0)

print("Converting principal point pixel coordinates to normalized coordinates: ")
p_pixels = Point2(320, 240)
print(f"Pixel point: {p_pixels}")

p_norm = cal.calibrate(p_pixels)
print(f"Calibrated (normalized) point: {p_norm}")

Converting principal point pixel coordinates to normalized coordinates: 
Pixel point: [320. 240.]
Calibrated (normalized) point: [0. 0.]


The `calibrate()` member function can optionally compute Jacobians with respect to the calibration parameters (`Dcal`) and the input point (`Dp`). This is useful for optimization tasks. Note that matrices you pass in must be column-major arrays with the correct shape.

In [15]:
# Jacobians for calibrate(p_pixels)
Dcal_calibrate = np.zeros((2, 1), order='F')
Dp_calibrate = np.zeros((2, 2), order='F')
p_norm = cal.calibrate(p_pixels, Dcal_calibrate, Dp_calibrate)
print(f"Jacobian Dcal_calibrate:\n{Dcal_calibrate}")
print(f"Jacobian Dp_calibrate:\n{Dp_calibrate}")

Jacobian Dcal_calibrate:
[[-0.]
 [-0.]]
Jacobian Dp_calibrate:
[[0.001 0.   ]
 [0.    0.001]]


### `uncalibrate()`

The `uncalibrate(p)` member function converts a 2D point `p` from normalized image coordinates $(x, y)$ back to image pixel coordinates $(u, v)$. This is the inverse operation of `calibrate()` and effectively implements the formula

$$
p_{\text{pixels}} = K \cdot p_{\text{norm}}
$$

In [16]:
p_pixels_recovered = cal.uncalibrate(p_norm)
print(f"Normalized point: {p_norm}")
print(f"Uncalibrated (pixel) point: {p_pixels_recovered}")

Normalized point: [0. 0.]
Uncalibrated (pixel) point: [320. 240.]


The `uncalibrate()` member function can also optionally compute Jacobians with respect to the calibration parameters (`Dcal`) and the input point (`Dp`). Likewise, the matrices you pass in must be column-major arrays with the correct shape.

In [17]:
# Jacobians for uncalibrate(p_norm)
Dcal_uncalibrate = np.zeros((2, 1), order='F')
Dp_uncalibrate = np.zeros((2, 2), order='F')
_ = cal.uncalibrate(p_norm, Dcal_uncalibrate, Dp_uncalibrate)
print(f"Jacobian Dcal_uncalibrate:\n{Dcal_uncalibrate}")
print(f"Jacobian Dp_uncalibrate:\n{Dp_uncalibrate}")

Jacobian Dcal_uncalibrate:
[[0.]
 [0.]]
Jacobian Dp_uncalibrate:
[[1000.    0.]
 [   0. 1000.]]


## Manifold Operations

`Cal3f`, like many geometric types in GTSAM, is treated as a manifold in order to optimize over it. This means it supports operations like `retract()` (moving on the manifold given a tangent vector) and `localCoordinates()` (finding the tangent vector between two points on the manifold). Since `Cal3f` has only one degree of freedom (the focal length), its tangent space is 1-dimensional.

In [18]:
print("Original cal_model:")
print(cal)

# Retract: Apply a delta to the calibration parameter
delta_vec = np.array([10.0])
cal_retracted = cal.retract(delta_vec)
print(f"Delta vector: {delta_vec}")
print("Retracted cal_retracted:")
print(cal_retracted)

# Local coordinates: Find the delta between two calibrations
local_coords = cal.localCoordinates(cal_retracted)
print("\nLocal coordinates from cal_model to cal_retracted:")
print(local_coords)

Original cal_model:
f: 1000, px: 320, py: 240

Delta vector: [10.]
Retracted cal_retracted:
f: 1010, px: 320, py: 240


Local coordinates from cal_model to cal_retracted:
[10.]


## Additional Resources

The following curated resources provide detailed explanations of the camera calibration process. If you found any parts of this user guide to be confusing, we recommend getting started by reviewing these resources first:

### Video(s)
- ["Linear Camera Model | Camera Calibration"](https://www.youtube.com/watch?v=1rhq4-wA3oY) by Dr. Shree Nayar from Columbia University
- ["Computer Vision: The Camera Matrix"](https://www.youtube.com/watch?v=S52s_1s-GeM) by Michael Prasthofer

## Source
- [Cal3f.h](https://github.com/borglab/gtsam/blob/develop/gtsam/geometry/Cal3f.h)
- [Cal3f.cpp](https://github.com/borglab/gtsam/blob/develop/gtsam/geometry/Cal3f.cpp)